#### 6.1. Verifying  libraries

In [5]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    try:
        v = version(lib)
        print(f'{lib}: {v}')
    except Exception as e:
        print(f'{lib}: Not found')

reasoning_from_scratch: Not found
torch: 2.4.1+cu124
tokenizers: 0.23.3


#### 6.2. Loading a pretrained model

In [6]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


In [14]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{21}

#### 6.3. Loading a MATH training subset

In [17]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/"
        "shubhamisok/math-500/resolve/main/"
        "math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        
        
        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [18]:
from pprint import pprint
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

#### 6.5. Sampling rollouts

In [19]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [23]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 To solve the equation:

\[
\frac{1}{2}(3x - 9) = x + 37
\]

**Step 1: Eliminate the fraction**

Multiply both sides by 2 to eliminate the fraction:

\[
2 \times \frac{1}{2}(3x - 9) = 2 \times (x + 37)
\]

Simplifying both sides:

\[
3x - 9 = 2x + 74
\]

**Step 2: Isolate the variable \( x \)**

Subtract \( 2x \) from both sides:

\[
3x - 2x - 9 = 74
\]

Simplify:

\[
x - 9 = 74
\]

**Step 3: Solve for \( x \)**

Add 9 to both sides:

\[
x = 74 + 9
\]

\[
x = 83
\]

**Final Answer:**

\[
\boxed{83}
\]<|endoftext|>


#### 6.5. Calculating rewards

In [25]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None
    )

    if not extracted:
        return 0.0

    correct = grade_answer(extracted, ground_truth)

    return float(correct)

In [26]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer, "83")
    rollout_rewards.append(reward)
    print(f"{answer!r} -> {reward}")

'\\boxed{83}' -> 1.0
'The correct answer is \\boxed{83}' -> 1.0
'The final answer is 83' -> 0.0
'We get \\boxed{38}' -> 0.0


#### 6.7. Preparing learning signals from rollouts via advantages

In [28]:
rewards = torch.tensor(rollout_rewards, device=device)

advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print("Rewards    :", rewards)
print("Advantages :", advantages)

Rewards    : tensor([1., 1., 0., 0.], device='cuda:0')
Advantages : tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')
